## simulating voting accuracy by p_correct

In [ ]:
p_raws = [0.1, 0.5, 0.7, 0.9]


def monte_carlo_majority_simulation(
    number_participants: int,
    p_correct: int,
    number_answers: int,
    num_simulations: int = 10000,
    random_seed: int = 42,
) -> float:
    np.random.seed(random_seed)

    successes = 0  # number of trials where majority is 1

    for _ in range(num_simulations):
        # if np.random.rand() > p_correct:
        #     continue

        p_correct_used = p_correct
        sampled = np.random.choice(
            np.arange(number_answers),
            size=number_participants,
            replace=True,
            p=[p_correct_used]
            + [
                (1 - p_correct_used) / (number_answers - 1)
                for _ in range(number_answers - 1)
            ],
        )

        c = np.unique_counts(sampled)

        most_taken = None
        most_taken_count = -1
        for c, v in zip(c.counts, c.values):
            if c > most_taken_count:
                most_taken_count = c
                most_taken = v
            elif c == most_taken_count:
                most_taken = None

        if most_taken == 0:
            successes += 1

    probability = successes / num_simulations

    return probability


def p_total(participants, p_raw: float):
    return np.array(
        [monte_carlo_majority_simulation(pi, p_raw, 10) for pi in participants]
    )


x = np.arange(1, 20)
plt.plot(x, np.array([p_total(x, p_raw) for p_raw in p_raws]).transpose())
plt.xticks(x)
plt.show()

In [ ]:
spread_info = (
    no_discussion_data.filter(pl.col("temperature") == 0)
    .with_columns(
        element_index=pl.col("compared_individual_answers").list.eval(
            pl.int_range(0, pl.len())
        )
    )
    .explode("compared_individual_answers", "element_index")
    .group_by("model_name", "element_index")
    .agg(
        pl.len(),
        pl.col("compared_individual_answers").mean().alias(plc.accuracy),
        pl.col(plc.used_input_tokens).mean(),
        pl.col(plc.used_output_tokens).mean(),
    )
)
spread_info

In [ ]:
value = "accuracy"

spread_info.group_by("model_name").agg(
    [
        pl.col(value).mean().alias("mean"),
        pl.col(value).std().alias("std"),
        pl.col(value).min().alias("min"),
        pl.col(value).quantile(0.25).alias("25%"),
        pl.col(value).median().alias("50%"),
        pl.col(value).quantile(0.75).alias("75%"),
        pl.col(value).max().alias("max"),
    ]
).with_columns(spread=pl.col("max") - pl.col("min")).with_columns(
    pl.col("model_name")
    .str.split("B")
    .list.first()
    .str.split("-")
    .list.last()
    .cast(float)
    .alias("params"),
    parse_model_family(pl.col("model_name")).alias("family"),
).sort("family", "params").drop("family", "params")